# Step 14: Human-in-the-Loop

        _Instructor Solution Notebook_  
        Estimated time: **75 minutes**  
        Difficulty: **Advanced**

        ## Learning Objectives
        - [ ] Pause a workflow for approval.
- [ ] Model resumable state with checkpoints.
- [ ] Treat human review as a deliberate system boundary.
- [ ] Design for long-running workflows that need supervision.

        ## Prerequisites
        - Step 13 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Human-in-the-loop is not just a UI prompt. It is a control point where the system explicitly waits for a trustworthy decision before acting.

Expected output and validation notes:

Expected output snapshot:

- An approval record is stored
- A checkpoint payload is available for resume


## Part 2: Core Implementation

### Approval and checkpoint helpers


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class ApprovalRequest:
    tool_name: str
    reason: str
    params: dict
    status: str = "pending"

@dataclass
class CheckpointStore:
    snapshots: dict = field(default_factory=dict)

    def save(self, checkpoint_id: str, payload: dict) -> None:
        self.snapshots[checkpoint_id] = {
            "saved_at": datetime.utcnow().isoformat(),
            "payload": payload,
        }

    def load(self, checkpoint_id: str) -> dict:
        return self.snapshots[checkpoint_id]["payload"]

approvals: list[ApprovalRequest] = []
checkpoint_store = CheckpointStore()


## Part 2: Core Implementation

### Simulate a gated operation


In [ ]:
async def request_approval(tool_name: str, reason: str, params: dict) -> bool:
    approval = ApprovalRequest(tool_name=tool_name, reason=reason, params=params)
    approvals.append(approval)
    approval.status = "approved" if tool_name != "delete_database" else "rejected"
    return approval.status == "approved"

async def protected_delete(dataset_id: str) -> str:
    allowed = await request_approval(
        tool_name="delete_dataset",
        reason="User requested destructive cleanup.",
        params={"dataset_id": dataset_id},
    )
    if not allowed:
        return "Deletion rejected."
    checkpoint_store.save("before_delete", {"dataset_id": dataset_id})
    return f"Deleted dataset {dataset_id}"

print(await protected_delete("archive_2020"))
print_json([approval.__dict__ for approval in approvals], title="Approvals")


## Part 3: Hands-On Exercises

Add a resume helper that restores a checkpoint and prints the last saved payload.

This mirrored notebook uses completed exercise code so instructors can demonstrate the target state.


In [ ]:
def resume_from(checkpoint_id: str) -> dict:
    return checkpoint_store.load(checkpoint_id)

print_json(resume_from("before_delete"), title="Resumed checkpoint payload")


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
def resume_from(checkpoint_id: str) -> dict:
    return checkpoint_store.load(checkpoint_id)

print_json(resume_from("before_delete"), title="Resumed checkpoint payload")


## Part 5: Best Practices & Tips

        - Require approval before destructive or irreversible actions.
- Checkpoint enough state to explain what the system was about to do.
- Separate approval records from model-generated text for auditability.


## Summary & Key Takeaways

You now have a basic pattern for approvals and checkpoint-based resume flows.


## Additional Resources

        - `Step 15 notebook`
- `docs/troubleshooting.md`
